In [ ]:
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

In [ ]:
# Find the project root from the notebook's current working directory.
working_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if any(
            (candidate / relative_path).is_dir()
            for relative_path in (
                Path("ai-service") / "data" / "voice",
                Path("ai-service") / "data" / "audio",
                Path("data") / "audio",
            )
        )
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find the project audio folders from the notebook's working directory."
    )

audio_directories = [
    project_root / "ai-service" / "data" / "voice",
    project_root / "ai-service" / "data" / "audio",
    project_root / "data" / "audio",
]
wav_files = sorted(
    (
        recording
        for directory in audio_directories
        if directory.is_dir()
        for recording in directory.rglob("*.wav")
        if recording.is_file()
    ),
    key=lambda recording: str(recording).casefold(),
)
print(f"Project root: {project_root}")
print(f"WAV recordings found: {len(wav_files)}")
if not wav_files:
    raise FileNotFoundError("No existing WAV recordings were found in the project audio folders.")
print("First recording:", wav_files[0].relative_to(project_root))

In [ ]:
selected_wav = wav_files[0]
audio, sample_rate = sf.read(selected_wav, always_2d=False)
duration_seconds = len(audio) / sample_rate if sample_rate else 0.0

In [ ]:
print(f"Filename: {selected_wav.name}")
print(f"Sample rate: {sample_rate} Hz")
print(f"Duration: {duration_seconds:.2f} seconds")
print(f"Audio shape: {audio.shape}")

In [ ]:
display(Audio(data=audio, rate=sample_rate))

In [ ]:
validation_results = {
    "file exists": selected_wav.is_file(),
    "WAV extension": selected_wav.suffix.casefold() == ".wav",
    "positive sample rate": sample_rate > 0,
    "audio is non-empty": audio.size > 0,
    "audio samples are finite": bool(np.isfinite(audio).all()),
    "duration is positive": duration_seconds > 0,
}
for check, passed in validation_results.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
assert all(validation_results.values()), "Audio validation failed; inspect the checks above."